In [ ]:
#!/usr/bin/env python3
# Heat-maps grouped by phylum — M9 run, mip & cohm excluded
# ──────────────────────────────────────────────────────────

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import matplotlib
matplotlib.use("cairo")          # high-quality output

# ──────────────────────────────────────────────────────────
# 1.  Read & clean matrices
# ──────────────────────────────────────────────────────────
file_paths = {
    "Amino Acids": "../../../data/graphing/NIS/average_similarity_Amino_AcidsPTYG.xlsx",
    "Cofactors":   "../../../data/graphing/NIS/average_similarity_CofactorsPTYG.xlsx",
    "Nucleotides": "../../../data/graphing/NIS/average_similarity_NucleotidesPTYG.xlsx",
    "Lipids":      "../../../data/graphing/NIS/average_similarity_LipidsPTYG.xlsx",
}

def clean_labels(df: pd.DataFrame) -> pd.DataFrame:
    """Strip 'Thermo…' and '…Vitconstrained' parts from labels."""
    return df.rename(
        index=lambda s: (s.replace("Thermo", "")
                           .replace("PTYGVitconstrained", "")
                           .replace("XyloseVitconstrained", "")),
        columns=lambda s: (s.replace("Thermo", "")
                             .replace("PTYGVitconstrained", "")
                             .replace("XyloseVitconstrained", "")),
    )


matrices = {
    name: clean_labels(pd.read_excel(path, index_col=0))
    for name, path in file_paths.items()
}

# ──────────────────────────────────────────────────────────
# 2.  Phylum look-ups & colour palette
# ──────────────────────────────────────────────────────────
phyla_dict = {
    "cohm": "Firmicutes", "cuvk": "Actinobacteria", "cxy":  "Actinobacteria",
    "mip":  "Actinobacteria", "rfa":  "Actinobacteria", "cpi": "Bacteroidetes",
    "flk":  "Bacteroidetes",  "mgk":  "Bacteroidetes",  "bbet": "Alphaproteobacteria",
    "buol": "Betaproteobacteria", "cauj": "Alphaproteobacteria", "dri": "Alphaproteobacteria",
    "lgu":  "Gammaproteobacteria", "lur":  "Gammaproteobacteria", "mlot": "Alphaproteobacteria",
    "pmor": "Gammaproteobacteria", "pscf": "Gammaproteobacteria", "pzu":  "Alphaproteobacteria",
    "racd": "Gammaproteobacteria", "tro":  "Alphaproteobacteria", "vpe":  "Betaproteobacteria",
}
phyla_order = [
    "Alphaproteobacteria", "Gammaproteobacteria", "Actinobacteria",
    "Bacteroidetes", "Betaproteobacteria", "Firmicutes", "Unknown",
]

palette      = sns.color_palette("Set2", len(phyla_order))
phyla_colors = {phy: palette[i] for i, phy in enumerate(phyla_order)}

def phylum_of(label: str) -> str:
    return phyla_dict.get(label[:4], "Unknown")

# ──────────────────────────────────────────────────────────
# 3.  Plot loop
# ──────────────────────────────────────────────────────────
for name, data in matrices.items():

    # 3-a. ordering: phylum block → alphabetical
    ordered_labels = sorted(
        data.index,
        key=lambda lab: (phyla_order.index(phylum_of(lab)), lab),
    )
    ordered_data = data.loc[ordered_labels, ordered_labels]

    # 3-b. colour side-bars
    row_colors = [phyla_colors[phylum_of(lab)] for lab in ordered_labels]

    # 3-c. choose cmap
    cmap = "Oranges" if name in {"Amino Acids", "Cofactors"} else "Blues"

    # 3-d. draw heat-map (clustering within phyla allowed)
    g = sns.clustermap(
        ordered_data,
        cmap=cmap, vmin=0, vmax=1,
        figsize=(10, 8),
        row_colors=row_colors, col_colors=row_colors,
        row_cluster=True, col_cluster=True,
        dendrogram_ratio=0.15,
        cbar_pos=(0.02, 0.8, 0.02, 0.15),
        xticklabels=False, yticklabels=False,          # custom labels later
    )

    # order seaborn settled on
    col_order = g.data2d.columns
    row_order = g.data2d.index
    centres   = np.arange(len(col_order)) + 0.5

    # ── TOP X-labels (under dendrogram) ────────────────────────────────────
    ax_top = g.ax_col_dendrogram
    ax_top.set_xticks(centres)
    ax_top.set_xticklabels(col_order, rotation=90, ha="center",
                            va="top", fontsize=6)
    ax_top.tick_params(axis="x", length=0)
    for t in ax_top.get_xticklabels():
        t.set_y(-0.02)

    # ── BOTTOM X-labels ───────────────────────────────────────────────────
    ax_hm = g.ax_heatmap
    ax_hm.set_xticks(centres)
    ax_hm.set_xticklabels(col_order, rotation=90, ha="center",
                          va="top", fontsize=6)
    ax_hm.tick_params(axis="x", bottom=True, top=False,
                      labelbottom=True, pad=2)

    # ── Y-labels (strain codes on both sides) ─────────────────────────────
    ax_hm.set_yticks(centres)
    ax_hm.set_yticklabels(row_order, rotation=0, va="center",
                          fontsize=6)
    ax_hm.tick_params(axis="y",
                      left=True,  labelleft=True,
                      right=True, labelright=True, pad=2)

    # ── EXTRA right-hand axis for phylum names ────────────────────────────
    ax_phy = ax_hm.twinx()
    ax_phy.set_ylim(ax_hm.get_ylim())
    ax_phy.set_yticks(centres)
    ax_phy.set_yticklabels([phylum_of(lab) for lab in row_order],
                           rotation=0, va="center", fontsize=6)
    ax_phy.tick_params(axis="y",
                       right=True,  labelright=True,
                       left=False,  labelleft=False,
                       pad=2)
    ax_phy.spines["right"].set_position(("axes", 1.10))
    ax_phy.spines["right"].set_visible(False)

    # ── Replace diagonal squares by the strain code ───────────────────────
    # For every row label find the matching column index
    for r_idx, r_lab in enumerate(row_order):
        if r_lab in col_order:
            c_idx = col_order.get_loc(r_lab)

            # 1. hide the original coloured square with a white rectangle
            rect = mpatches.Rectangle(
                (c_idx, r_idx),          # (x, y) lower-left
                width=1, height=1,
                facecolor="white",
                edgecolor="none",
                zorder=3                # above heat-map, below text
            )
            ax_hm.add_patch(rect)

            # 2. write the strain code centred in that cell
            ax_hm.text(c_idx + 0.5, r_idx + 0.5, r_lab,
                       ha="center", va="center",
                       fontsize=6, fontweight="bold",
                       zorder=4)

    # ── Legend (on the dendrogram axis) ───────────────────────────────────
    for phy in phyla_order:
        ax_top.bar(0, 0, color=phyla_colors[phy], label=phy, linewidth=0)
    ax_top.legend(loc="center", ncol=3,
                  bbox_to_anchor=(0.5, 1.25), frameon=False)

    # ── save ───────────────────────────────────────────────────────────────
    pdf_name = f"{name.replace(' ', '_')}_PTYG_heatmap.pdf"
    g.figure.savefig(pdf_name, dpi=300, bbox_inches="tight")
    print(f"Saved {pdf_name}")
    plt.show()


In [ ]:
#!/usr/bin/env python3
# Heat-maps grouped by phylum — M9 run, mip & cohm excluded
# ──────────────────────────────────────────────────────────

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import matplotlib
matplotlib.use("cairo")          # high-quality output

# ──────────────────────────────────────────────────────────
# 1.  Read & clean matrices
# ──────────────────────────────────────────────────────────
file_paths = {
    "Amino Acids": "../../../data/graphing/NIS/average_similarity_Amino_AcidsM9.xlsx",
    "Cofactors":   "../../../data/graphing/NIS/average_similarity_CofactorsM9.xlsx",
    "Nucleotides": "../../../data/graphing/NIS/average_similarity_NucleotidesM9.xlsx",
    "Lipids":      "../../../data/graphing/NIS/average_similarity_LipidsM9.xlsx",
}

def clean_labels(df: pd.DataFrame) -> pd.DataFrame:
    """Strip 'Thermo…' and '…Vitconstrained' parts from labels."""
    return df.rename(
        index=lambda s: (s.replace("Thermo", "")
                           .replace("SuccinateVitconstrained", "")
                           .replace("XyloseVitconstrained", "")),
        columns=lambda s: (s.replace("Thermo", "")
                             .replace("SuccinateVitconstrained", "")
                             .replace("XyloseVitconstrained", "")),
    )

# drop mip & cohm everywhere
missing_codes = {"mip", "cohm"}

def drop_missing(df: pd.DataFrame) -> pd.DataFrame:
    keep = [lab for lab in df.index
            if lab[:4] not in missing_codes and lab[:3] not in missing_codes]
    return df.loc[keep, keep]

matrices = {
    name: drop_missing(clean_labels(pd.read_excel(path, index_col=0)))
    for name, path in file_paths.items()
}

# ──────────────────────────────────────────────────────────
# 2.  Phylum look-ups & colour palette
# ──────────────────────────────────────────────────────────
phyla_dict = {
    "cohm": "Firmicutes", "cuvk": "Actinobacteria", "cxy":  "Actinobacteria",
    "mip":  "Actinobacteria", "rfa":  "Actinobacteria", "cpi": "Bacteroidetes",
    "flk":  "Bacteroidetes",  "mgk":  "Bacteroidetes",  "bbet": "Alphaproteobacteria",
    "buol": "Betaproteobacteria", "cauj": "Alphaproteobacteria", "dri": "Alphaproteobacteria",
    "lgu":  "Gammaproteobacteria", "lur":  "Gammaproteobacteria", "mlot": "Alphaproteobacteria",
    "pmor": "Gammaproteobacteria", "pscf": "Gammaproteobacteria", "pzu":  "Alphaproteobacteria",
    "racd": "Gammaproteobacteria", "tro":  "Alphaproteobacteria", "vpe":  "Betaproteobacteria",
}
phyla_order = [
    "Alphaproteobacteria", "Gammaproteobacteria", "Actinobacteria",
    "Bacteroidetes", "Betaproteobacteria", "Firmicutes", "Unknown",
]

palette      = sns.color_palette("Set2", len(phyla_order))
phyla_colors = {phy: palette[i] for i, phy in enumerate(phyla_order)}

def phylum_of(label: str) -> str:
    return phyla_dict.get(label[:4], "Unknown")

# ──────────────────────────────────────────────────────────
# 3.  Plot loop
# ──────────────────────────────────────────────────────────
for name, data in matrices.items():

    # 3-a. ordering: phylum block → alphabetical
    ordered_labels = sorted(
        data.index,
        key=lambda lab: (phyla_order.index(phylum_of(lab)), lab),
    )
    ordered_data = data.loc[ordered_labels, ordered_labels]

    # 3-b. colour side-bars
    row_colors = [phyla_colors[phylum_of(lab)] for lab in ordered_labels]

    # 3-c. choose cmap
    cmap = "Oranges" if name in {"Amino Acids", "Cofactors"} else "Blues"

    # 3-d. draw heat-map (clustering within phyla allowed)
    g = sns.clustermap(
        ordered_data,
        cmap=cmap, vmin=0, vmax=1,
        figsize=(10, 8),
        row_colors=row_colors, col_colors=row_colors,
        row_cluster=True, col_cluster=True,
        dendrogram_ratio=0.15,
        cbar_pos=(0.02, 0.8, 0.02, 0.15),
        xticklabels=False, yticklabels=False,          # custom labels later
    )

    # order seaborn settled on
    col_order = g.data2d.columns
    row_order = g.data2d.index
    centres   = np.arange(len(col_order)) + 0.5

    # ── TOP X-labels (under dendrogram) ────────────────────────────────────
    ax_top = g.ax_col_dendrogram
    ax_top.set_xticks(centres)
    ax_top.set_xticklabels(col_order, rotation=90, ha="center",
                            va="top", fontsize=6)
    ax_top.tick_params(axis="x", length=0)
    for t in ax_top.get_xticklabels():
        t.set_y(-0.02)

    # ── BOTTOM X-labels ───────────────────────────────────────────────────
    ax_hm = g.ax_heatmap
    ax_hm.set_xticks(centres)
    ax_hm.set_xticklabels(col_order, rotation=90, ha="center",
                          va="top", fontsize=6)
    ax_hm.tick_params(axis="x", bottom=True, top=False,
                      labelbottom=True, pad=2)

    # ── Y-labels (strain codes on both sides) ─────────────────────────────
    ax_hm.set_yticks(centres)
    ax_hm.set_yticklabels(row_order, rotation=0, va="center",
                          fontsize=6)
    ax_hm.tick_params(axis="y",
                      left=True,  labelleft=True,
                      right=True, labelright=True, pad=2)

    # ── EXTRA right-hand axis for phylum names ────────────────────────────
    ax_phy = ax_hm.twinx()
    ax_phy.set_ylim(ax_hm.get_ylim())
    ax_phy.set_yticks(centres)
    ax_phy.set_yticklabels([phylum_of(lab) for lab in row_order],
                           rotation=0, va="center", fontsize=6)
    ax_phy.tick_params(axis="y",
                       right=True,  labelright=True,
                       left=False,  labelleft=False,
                       pad=2)
    ax_phy.spines["right"].set_position(("axes", 1.10))
    ax_phy.spines["right"].set_visible(False)

    # ── Replace diagonal squares by the strain code ───────────────────────
    # For every row label find the matching column index
    for r_idx, r_lab in enumerate(row_order):
        if r_lab in col_order:
            c_idx = col_order.get_loc(r_lab)

            # 1. hide the original coloured square with a white rectangle
            rect = mpatches.Rectangle(
                (c_idx, r_idx),          # (x, y) lower-left
                width=1, height=1,
                facecolor="white",
                edgecolor="none",
                zorder=3                # above heat-map, below text
            )
            ax_hm.add_patch(rect)

            # 2. write the strain code centred in that cell
            ax_hm.text(c_idx + 0.5, r_idx + 0.5, r_lab,
                       ha="center", va="center",
                       fontsize=6, fontweight="bold",
                       zorder=4)

    # ── Legend (on the dendrogram axis) ───────────────────────────────────
    for phy in phyla_order:
        ax_top.bar(0, 0, color=phyla_colors[phy], label=phy, linewidth=0)
    ax_top.legend(loc="center", ncol=3,
                  bbox_to_anchor=(0.5, 1.25), frameon=False)

    # ── save ───────────────────────────────────────────────────────────────
    pdf_name = f"{name.replace(' ', '_')}_M9_heatmap.pdf"
    g.figure.savefig(pdf_name, dpi=300, bbox_inches="tight")
    print(f"Saved {pdf_name}")
    plt.show()


In [ ]:
#!/usr/bin/env python3
# Circular Jaccard diagram grouped by phylum
# Node order: Firmicutes-Beta-Bacteroidetes-Actino-Gamma-Alpha
# Edges = pairs with Jaccard ≥ 0.8, colored by metric

import math
import pandas as pd
import numpy as np
import matplotlib
         # high-quality vector output
import matplotlib.pyplot as plt
from matplotlib.path import Path
from matplotlib.patches import PathPatch, Circle
import seaborn as sns
matplotlib.use("cairo")          # high-quality vector output
# ──────────────────────────────────────────────────────
# 1. Read & clean matrices
# ──────────────────────────────────────────────────────
file_paths = {
    "Amino Acids": "../../../data/graphing/NIS/average_similarity_Amino_AcidsM9.xlsx",
    "Cofactors":   "../../../data/graphing/NIS/average_similarity_CofactorsM9.xlsx",
    "Nucleotides": "../../../data/graphing/NIS/average_similarity_NucleotidesM9.xlsx",
    "Lipids":      "../../../data/graphing/NIS/average_similarity_LipidsM9.xlsx",
}

def clean_labels(df: pd.DataFrame) -> pd.DataFrame:
    """Strip 'Thermo…' and '…Vitconstrained' parts from labels."""
    return df.rename(
        index=lambda s: (s.replace("Thermo", "")
                           .replace("SuccinateVitconstrained", "")
                           .replace("XyloseVitconstrained", "")),
        columns=lambda s: (s.replace("Thermo", "")
                             .replace("SuccinateVitconstrained", "")
                             .replace("XyloseVitconstrained", "")),
    )

# Drop mip & cohm everywhere
missing_codes = {"mip", "cohm"}

def drop_missing(df: pd.DataFrame) -> pd.DataFrame:
    keep = [
        lab for lab in df.index
        if lab[:4] not in missing_codes and lab[:3] not in missing_codes
    ]
    return df.loc[keep, keep]

matrices = {
    name: drop_missing(clean_labels(pd.read_excel(path, index_col=0)))
    for name, path in file_paths.items()
}

# ──────────────────────────────────────────────────────
# 2. Phylum mapping & desired order
# ──────────────────────────────────────────────────────
phyla_dict = {
    "cohm": "Firmicutes", "cuvk": "Actinobacteria", "cxy":  "Actinobacteria",
    "mip":  "Actinobacteria", "rfa":  "Actinobacteria", "cpi": "Bacteroidetes",
    "flk":  "Bacteroidetes",  "mgk":  "Bacteroidetes",  "bbet": "Alphaproteobacteria",
    "buol": "Betaproteobacteria", "cauj": "Alphaproteobacteria", "dri": "Alphaproteobacteria",
    "lgu":  "Gammaproteobacteria", "lur":  "Gammaproteobacteria", "mlot": "Alphaproteobacteria",
    "pmor": "Gammaproteobacteria", "pscf": "Gammaproteobacteria", "pzu":  "Alphaproteobacteria",
    "racd": "Gammaproteobacteria", "tro":  "Alphaproteobacteria", "vpe":  "Betaproteobacteria",
}

# Your requested order:
phyla_order = [
    "Firmicutes",
    "Betaproteobacteria",
    "Bacteroidetes",
    "Actinobacteria",
    "Gammaproteobacteria",
    "Alphaproteobacteria",
    "Unknown",  # fallback bucket
]

palette      = sns.color_palette("Set2", len(phyla_order))
phyla_colors = {phy: palette[i] for i, phy in enumerate(phyla_order)}

def phylum_of(label: str) -> str:
    return phyla_dict.get(label[:4], "Unknown")

# ──────────────────────────────────────────────────────
# 3. Determine common labels & ordering (by phylum, then name)
# ──────────────────────────────────────────────────────
label_sets = [set(df.index) for df in matrices.values()]
common_labels = sorted(set.intersection(*label_sets))

if not common_labels:
    raise ValueError("No common labels found across all matrices after cleaning.")

# Order by phylum block (your order), then alphabetically within block
ordered_labels = sorted(
    common_labels,
    key=lambda lab: (phyla_order.index(phylum_of(lab)), lab),
)

n = len(ordered_labels)

# Map labels to angles on the circle
angles = {lab: 2 * math.pi * i / n for i, lab in enumerate(ordered_labels)}

# ──────────────────────────────────────────────────────
# 4. Build edge list from all matrices (Jaccard ≥ 0.8)
# ──────────────────────────────────────────────────────
metric_colors = {
    "Amino Acids": "tab:orange",
    "Cofactors":   "tab:green",
    "Nucleotides": "tab:blue",
    "Lipids":      "tab:red",
}

threshold = 0.6  # Jaccard cut-off

edges = []  # list of dicts: {"source", "target", "value", "metric"}

for metric_name, df in matrices.items():
    df = df.loc[ordered_labels, ordered_labels]

    for i, src in enumerate(ordered_labels):
        for j in range(i + 1, n):  # upper triangle only
            tgt = ordered_labels[j]
            val = df.iloc[i, j]
            if pd.notna(val) and val >= threshold:
                edges.append({
                    "source": src,
                    "target": tgt,
                    "value": float(val),
                    "metric": metric_name,
                })

print(f"Number of edges with Jaccard ≥ {threshold}: {len(edges)}")

# ──────────────────────────────────────────────────────
# 5. Plot: circular nodes + curved edges
# ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 10))
ax.set_aspect("equal")
ax.axis("off")

R_nodes = 1.0     # radius for nodes
R_control = 0.7   # radius for control points of curves

def polar_to_cartesian(r, theta):
    """Convert polar (r, theta) to Cartesian (x, y)."""
    return r * math.cos(theta), r * math.sin(theta)

# 5-a. Draw nodes on circle (colored by phylum)
for lab in ordered_labels:
    theta = angles[lab]
    x, y = polar_to_cartesian(R_nodes, theta)
    ax.scatter(
        x, y,
        s=20,
        color=phyla_colors[phylum_of(lab)],
        edgecolor="black",
        linewidth=0.5,
        zorder=5,
    )

# 5-a-bis. Draw the outer circle
circle = Circle(
    (0, 0), R_nodes,
    edgecolor="black",
    facecolor="none",
    linewidth=1,
    zorder=1,
)
ax.add_patch(circle)

# 5-b. Label nodes (species codes) around the circle
for lab in ordered_labels:
    theta = angles[lab]
    x, y = polar_to_cartesian(R_nodes * 1.08, theta)  # slightly outside
    rotation = math.degrees(theta)

    # Flip text orientation on left half so it's readable
    if 90 < rotation < 270:
        rotation += 180
        ha = "right"
    else:
        ha = "left"

    ax.text(
        x, y, lab,
        fontsize=6,
        rotation=rotation,
        rotation_mode="anchor",
        ha=ha,
        va="center",
        zorder=6,
    )

# 5-c. Draw curved links (edges)
min_lw = 0.5
max_lw = 4.0

def scale_lw(val, vmin=threshold, vmax=1.0):
    """Scale line width between min_lw and max_lw based on Jaccard value."""
    if vmax == vmin:
        return (min_lw + max_lw) / 2
    t = (val - vmin) / (vmax - vmin)
    t = max(0.0, min(1.0, t))
    return min_lw + t * (max_lw - min_lw)

for edge in edges:
    src = edge["source"]
    tgt = edge["target"]
    val = edge["value"]
    metric = edge["metric"]

    theta_src = angles[src]
    theta_tgt = angles[tgt]

    x0, y0 = polar_to_cartesian(R_nodes, theta_src)
    x1, y1 = polar_to_cartesian(R_nodes, theta_tgt)

    # Control point: mid-angle, closer to centre
    theta_mid = (theta_src + theta_tgt) / 2.0

    # Handle wrap-around across 0/2π
    if abs(theta_src - theta_tgt) > math.pi:
        theta_mid += math.pi

    x_ctrl, y_ctrl = polar_to_cartesian(R_control, theta_mid)

    verts = [(x0, y0), (x_ctrl, y_ctrl), (x1, y1)]
    codes = [Path.MOVETO, Path.CURVE3, Path.CURVE3]

    path = Path(verts, codes)
    patch = PathPatch(
        path,
        lw=scale_lw(val),
        edgecolor=metric_colors[metric],
        facecolor="none",
        alpha=0.6,
        zorder=2,
    )
    ax.add_patch(patch)

# 5-d. Legends: metrics (edge colors) + phyla (node colors)
metric_handles = [
    plt.Line2D([0], [0], color=c, lw=3, label=m)
    for m, c in metric_colors.items()
]

phylum_handles = [
    plt.Line2D(
        [0], [0],
        marker="o", linestyle="",
        color=phyla_colors[phy],
        label=phy,
        markersize=6,
    )
    for phy in phyla_order
]

leg1 = ax.legend(
    handles=metric_handles,
    title="Metric",
    loc="upper left",
    bbox_to_anchor=(0.02, 0.98),
    frameon=False,
)
ax.add_artist(leg1)

ax.legend(
    handles=phylum_handles,
    title="Phylum",
    loc="upper right",
    bbox_to_anchor=(0.98, 0.98),
    frameon=False,
)

fig.tight_layout()
fig.savefig("circular_jaccard_grouped_ge_0_8.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
#!/usr/bin/env python3
# Four circular Jaccard diagrams, one per metric
# Node order: Firmicutes-Beta-Bacteroidetes-Actino-Gamma-Alpha
# Edges = pairs with Jaccard ≥ threshold
# Higher Jaccard -> more saturated red (A01821-based gradient)

import math
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.path import Path
from matplotlib.patches import PathPatch, Circle
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
import seaborn as sns

# ──────────────────────────────────────────────────────
# 1. Read & clean matrices
# ──────────────────────────────────────────────────────
file_paths = {
    "Amino Acids": "../../../data/graphing/NIS/average_similarity_Amino_AcidsM9.xlsx",
    "Cofactors":   "../../../data/graphing/NIS/average_similarity_CofactorsM9.xlsx",
    "Nucleotides": "../../../data/graphing/NIS/average_similarity_NucleotidesM9.xlsx",
    "Lipids":      "../../../data/graphing/NIS/average_similarity_LipidsM9.xlsx",
}

def clean_labels(df: pd.DataFrame) -> pd.DataFrame:
    """Strip 'Thermo…' and '…Vitconstrained' parts from labels."""
    return df.rename(
        index=lambda s: (s.replace("Thermo", "")
                           .replace("SuccinateVitconstrained", "")
                           .replace("XyloseVitconstrained", "")),
        columns=lambda s: (s.replace("Thermo", "")
                             .replace("SuccinateVitconstrained", "")
                             .replace("XyloseVitconstrained", "")),
    )

missing_codes = {"mip", "cohm"}

def drop_missing(df: pd.DataFrame) -> pd.DataFrame:
    keep = [
        lab for lab in df.index
        if lab[:4] not in missing_codes and lab[:3] not in missing_codes
    ]
    return df.loc[keep, keep]

matrices = {
    name: drop_missing(clean_labels(pd.read_excel(path, index_col=0)))
    for name, path in file_paths.items()
}

# ──────────────────────────────────────────────────────
# 2. Phylum mapping & order
# ──────────────────────────────────────────────────────
phyla_dict = {
    "cohm": "Firmicutes", "cuvk": "Actinobacteria", "cxy":  "Actinobacteria",
    "mip":  "Actinobacteria", "rfa":  "Actinobacteria", "cpi": "Bacteroidetes",
    "flk":  "Bacteroidetes",  "mgk":  "Bacteroidetes",  "bbet": "Alphaproteobacteria",
    "buol": "Betaproteobacteria", "cauj": "Alphaproteobacteria", "dri": "Alphaproteobacteria",
    "lgu":  "Gammaproteobacteria", "lur":  "Gammaproteobacteria", "mlot": "Alphaproteobacteria",
    "pmor": "Gammaproteobacteria", "pscf": "Gammaproteobacteria", "pzu":  "Alphaproteobacteria",
    "racd": "Gammaproteobacteria", "tro":  "Alphaproteobacteria", "vpe":  "Betaproteobacteria",
}

phyla_order = [
    "Firmicutes",
    "Betaproteobacteria",
    "Bacteroidetes",
    "Actinobacteria",
    "Gammaproteobacteria",
    "Alphaproteobacteria",
    "Unknown",
]

palette      = sns.color_palette("Set2", len(phyla_order))
phyla_colors = {phy: palette[i] for i, phy in enumerate(phyla_order)}

def phylum_of(label: str) -> str:
    return phyla_dict.get(label[:4], "Unknown")

# ──────────────────────────────────────────────────────
# 3. Common labels & circular ordering
# ──────────────────────────────────────────────────────
label_sets = [set(df.index) for df in matrices.values()]
common_labels = sorted(set.intersection(*label_sets))
if not common_labels:
    raise ValueError("No common labels found across all matrices after cleaning.")

ordered_labels = sorted(
    common_labels,
    key=lambda lab: (phyla_order.index(phylum_of(lab)), lab),
)

n = len(ordered_labels)
angles = {lab: 2 * math.pi * i / n for i, lab in enumerate(ordered_labels)}

# ──────────────────────────────────────────────────────
# 4. Helper functions & Jaccard colormap
# ──────────────────────────────────────────────────────
threshold = 0.6
edge_lw = 1.5                      # constant stroke width

def polar_to_cartesian(r, theta):
    return r * math.cos(theta), r * math.sin(theta)

# Build a sequential red scale from very light → A01821
base_red = "#A01821"
cmap_edges = sns.light_palette(base_red, n_colors=256, as_cmap=True)
norm = mcolors.Normalize(vmin=threshold, vmax=1.0)

# ──────────────────────────────────────────────────────
# 5. Loop over metrics and draw the 4 panels
# ──────────────────────────────────────────────────────
R_nodes = 1.0
R_control = 0.7

for metric_name, df in matrices.items():
    df = df.loc[ordered_labels, ordered_labels]

    edges = []
    for i, src in enumerate(ordered_labels):
        for j in range(i + 1, n):
            tgt = ordered_labels[j]
            val = df.iloc[i, j]
            if pd.notna(val) and val >= threshold:
                edges.append((src, tgt, float(val)))

    print(f"{metric_name}: {len(edges)} edges with Jaccard ≥ {threshold}")

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_aspect("equal")
    ax.axis("off")

    # outer circle
    circle = Circle((0, 0), R_nodes,
                    edgecolor="black", facecolor="none",
                    linewidth=1, zorder=1)
    ax.add_patch(circle)

    # nodes
    for lab in ordered_labels:
        theta = angles[lab]
        x, y = polar_to_cartesian(R_nodes, theta)
        ax.scatter(
            x, y,
            s=20,
            color=phyla_colors[phylum_of(lab)],
            edgecolor="black",
            linewidth=0.5,
            zorder=5,
        )

    # labels
    for lab in ordered_labels:
        theta = angles[lab]
        x, y = polar_to_cartesian(R_nodes * 1.08, theta)
        rot = math.degrees(theta)
        if 90 < rot < 270:
            rot += 180
            ha = "right"
        else:
            ha = "left"

        ax.text(
            x, y, lab,
            fontsize=6,
            rotation=rot,
            rotation_mode="anchor",
            ha=ha,
            va="center",
            zorder=6,
        )

    # edges coloured by Jaccard (pink-red gradient)
    for src, tgt, val in edges:
        theta_src = angles[src]
        theta_tgt = angles[tgt]
        x0, y0 = polar_to_cartesian(R_nodes, theta_src)
        x1, y1 = polar_to_cartesian(R_nodes, theta_tgt)

        theta_mid = (theta_src + theta_tgt) / 2.0
        if abs(theta_src - theta_tgt) > math.pi:
            theta_mid += math.pi
        x_ctrl, y_ctrl = polar_to_cartesian(R_control, theta_mid)

        verts = [(x0, y0), (x_ctrl, y_ctrl), (x1, y1)]
        codes = [Path.MOVETO, Path.CURVE3, Path.CURVE3]

        color = cmap_edges(norm(val))

        path = Path(verts, codes)
        patch = PathPatch(
            path,
            lw=edge_lw,
            edgecolor=color,
            facecolor="none",
            alpha=1.0,
            zorder=2,
        )
        ax.add_patch(patch)

    # Phylum legend
    phylum_handles = [
        Line2D(
            [0], [0],
            marker="o", linestyle="",
            color=phyla_colors[phy],
            label=phy,
            markersize=6,
        )
        for phy in phyla_order
    ]
    phylum_leg = ax.legend(
        handles=phylum_handles,
        title="Phylum",
        loc="upper right",
        bbox_to_anchor=(0.98, 0.98),
        frameon=False,
    )
    ax.add_artist(phylum_leg)

    # Colourbar for Jaccard
    sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap_edges)
    sm.set_array([])
    cbar = fig.colorbar(
        sm, ax=ax, fraction=0.03, pad=0.04
    )
    cbar.set_label("Jaccard similarity", fontsize=8)
    cbar.set_ticks([threshold, 0.8, 1.0])

    ax.set_title(f"{metric_name} (Jaccard ≥ {threshold})", fontsize=12)

    fig.tight_layout()
    safe_name = metric_name.replace(" ", "_")
    fig.savefig(
        f"circular_jaccard_{safe_name}_ge_{str(threshold).replace('.', '')}.pdf",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()


In [ ]:
#!/usr/bin/env python3
# Four circular Jaccard diagrams, one per metric
# Node order: Firmicutes-Beta-Bacteroidetes-Actino-Gamma-Alpha
# For each cutoff in [0.2, 0.4, 0.8], plot edges with Jaccard ≥ cutoff
# Higher Jaccard -> more saturated red (A01821-based gradient)
# Colourbar for each panel runs from cutoff -> 1.0

import math
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.path import Path
from matplotlib.patches import PathPatch, Circle
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
import seaborn as sns

# ──────────────────────────────────────────────────────
# 1. Read & clean matrices
# ──────────────────────────────────────────────────────
file_paths = {
    "Amino Acids": "../../../data/graphing/NIS/average_similarity_Amino_AcidsM9.xlsx",
    "Cofactors":   "../../../data/graphing/NIS/average_similarity_CofactorsM9.xlsx",
    "Nucleotides": "../../../data/graphing/NIS/average_similarity_NucleotidesM9.xlsx",
    "Lipids":      "../../../data/graphing/NIS/average_similarity_LipidsM9.xlsx",
}

def clean_labels(df: pd.DataFrame) -> pd.DataFrame:
    """Strip 'Thermo…' and '…Vitconstrained' parts from labels."""
    return df.rename(
        index=lambda s: (s.replace("Thermo", "")
                           .replace("SuccinateVitconstrained", "")
                           .replace("XyloseVitconstrained", "")),
        columns=lambda s: (s.replace("Thermo", "")
                             .replace("SuccinateVitconstrained", "")
                             .replace("XyloseVitconstrained", "")),
    )

missing_codes = {"mip", "cohm"}

def drop_missing(df: pd.DataFrame) -> pd.DataFrame:
    keep = [
        lab for lab in df.index
        if lab[:4] not in missing_codes and lab[:3] not in missing_codes
    ]
    return df.loc[keep, keep]

matrices = {
    name: drop_missing(clean_labels(pd.read_excel(path, index_col=0)))
    for name, path in file_paths.items()
}

# ──────────────────────────────────────────────────────
# 2. Phylum mapping & order
# ──────────────────────────────────────────────────────
phyla_dict = {
    "cohm": "Firmicutes", "cuvk": "Actinobacteria", "cxy":  "Actinobacteria",
    "mip":  "Actinobacteria", "rfa":  "Actinobacteria", "cpi": "Bacteroidetes",
    "flk":  "Bacteroidetes",  "mgk":  "Bacteroidetes",  "bbet": "Alphaproteobacteria",
    "buol": "Betaproteobacteria", "cauj": "Alphaproteobacteria", "dri": "Alphaproteobacteria",
    "lgu":  "Gammaproteobacteria", "lur":  "Gammaproteobacteria", "mlot": "Alphaproteobacteria",
    "pmor": "Gammaproteobacteria", "pscf": "Gammaproteobacteria", "pzu":  "Alphaproteobacteria",
    "racd": "Gammaproteobacteria", "tro":  "Alphaproteobacteria", "vpe":  "Betaproteobacteria",
}

phyla_order = [
    "Firmicutes",
    "Betaproteobacteria",
    "Bacteroidetes",
    "Actinobacteria",
    "Gammaproteobacteria",
    "Alphaproteobacteria",
    "Unknown",
]

palette      = sns.color_palette("Set2", len(phyla_order))
phyla_colors = {phy: palette[i] for i, phy in enumerate(phyla_order)}

def phylum_of(label: str) -> str:
    return phyla_dict.get(label[:4], "Unknown")

# ──────────────────────────────────────────────────────
# 3. Common labels & circular ordering
# ──────────────────────────────────────────────────────
label_sets = [set(df.index) for df in matrices.values()]
common_labels = sorted(set.intersection(*label_sets))
if not common_labels:
    raise ValueError("No common labels found across all matrices after cleaning.")

ordered_labels = sorted(
    common_labels,
    key=lambda lab: (phyla_order.index(phylum_of(lab)), lab),
)

n = len(ordered_labels)
angles = {lab: 2 * math.pi * i / n for i, lab in enumerate(ordered_labels)}

# ──────────────────────────────────────────────────────
# 4. Helper functions & base colormap
# ──────────────────────────────────────────────────────
cutoffs = [0.2, 0.4, 0.8]     # thresholds we want
edge_lw = 1.5                 # constant stroke width

def polar_to_cartesian(r, theta):
    return r * math.cos(theta), r * math.sin(theta)

# Build a sequential red scale from very light → A01821
base_red = "#A01821"
cmap_edges = sns.light_palette(base_red, n_colors=256, as_cmap=True)

# ──────────────────────────────────────────────────────
# 5. Loop over cutoffs and metrics
# ──────────────────────────────────────────────────────
R_nodes = 1.0
R_control = 0.7

for cutoff in cutoffs:
    # normalisation (colour scale) specific to this cutoff
    norm = mcolors.Normalize(vmin=cutoff, vmax=1.0)

    for metric_name, df in matrices.items():
        df = df.loc[ordered_labels, ordered_labels]

        edges = []
        for i, src in enumerate(ordered_labels):
            for j in range(i + 1, n):
                tgt = ordered_labels[j]
                val = df.iloc[i, j]
                if pd.notna(val) and val >= cutoff:
                    edges.append((src, tgt, float(val)))

        print(f"cutoff={cutoff}, {metric_name}: {len(edges)} edges with Jaccard ≥ {cutoff}")

        fig, ax = plt.subplots(figsize=(10, 10))
        ax.set_aspect("equal")
        ax.axis("off")

        # outer circle
        circle = Circle((0, 0), R_nodes,
                        edgecolor="black", facecolor="none",
                        linewidth=1, zorder=1)
        ax.add_patch(circle)

        # nodes
        for lab in ordered_labels:
            theta = angles[lab]
            x, y = polar_to_cartesian(R_nodes, theta)
            ax.scatter(
                x, y,
                s=20,
                color=phyla_colors[phylum_of(lab)],
                edgecolor="black",
                linewidth=0.5,
                zorder=5,
            )

        # labels
        for lab in ordered_labels:
            theta = angles[lab]
            x, y = polar_to_cartesian(R_nodes * 1.08, theta)
            rot = math.degrees(theta)
            if 90 < rot < 270:
                rot += 180
                ha = "right"
            else:
                ha = "left"

            ax.text(
                x, y, lab,
                fontsize=6,
                rotation=rot,
                rotation_mode="anchor",
                ha=ha,
                va="center",
                zorder=6,
            )

        # edges coloured by Jaccard (pink-red gradient)
        for src, tgt, val in edges:
            theta_src = angles[src]
            theta_tgt = angles[tgt]
            x0, y0 = polar_to_cartesian(R_nodes, theta_src)
            x1, y1 = polar_to_cartesian(R_nodes, theta_tgt)

            theta_mid = (theta_src + theta_tgt) / 2.0
            if abs(theta_src - theta_tgt) > math.pi:
                theta_mid += math.pi
            x_ctrl, y_ctrl = polar_to_cartesian(R_control, theta_mid)

            verts = [(x0, y0), (x_ctrl, y_ctrl), (x1, y1)]
            codes = [Path.MOVETO, Path.CURVE3, Path.CURVE3]

            color = cmap_edges(norm(val))

            path = Path(verts, codes)
            patch = PathPatch(
                path,
                lw=edge_lw,
                edgecolor=color,
                facecolor="none",
                alpha=1.0,
                zorder=2,
            )
            ax.add_patch(patch)

        # Phylum legend
        phylum_handles = [
            Line2D(
                [0], [0],
                marker="o", linestyle="",
                color=phyla_colors[phy],
                label=phy,
                markersize=6,
            )
            for phy in phyla_order
        ]
        phylum_leg = ax.legend(
            handles=phylum_handles,
            title="Phylum",
            loc="upper right",
            bbox_to_anchor=(0.98, 0.98),
            frameon=False,
        )
        ax.add_artist(phylum_leg)

        # Colourbar for Jaccard (cutoff → 1.0)
        sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap_edges)
        sm.set_array([])
        cbar = fig.colorbar(
            sm, ax=ax, fraction=0.03, pad=0.04
        )
        cbar.set_label("Jaccard similarity", fontsize=8)

        # ticks spaced between cutoff and 1
        tick_vals = [cutoff, (cutoff + 1.0) / 2.0, 1.0]
        cbar.set_ticks(tick_vals)
        cbar.set_ticklabels([f"{v:.2f}" for v in tick_vals])

        ax.set_title(f"{metric_name} (Jaccard ≥ {cutoff})", fontsize=12)

        fig.tight_layout()
        safe_name = metric_name.replace(" ", "_")
        thr_tag = f"{int(cutoff*100):02d}"   # 0.2 -> '20', 0.4 -> '40', 0.8 -> '80'
        fig.savefig(
            f"circular_jaccard_{safe_name}_ge_{thr_tag}.pdf",
            dpi=300,
            bbox_inches="tight",
        )
        plt.show()


In [ ]:
#!/usr/bin/env python3
# Four circular Jaccard diagrams, one per metric
# Node order: Firmicutes-Beta-Bacteroidetes-Actino-Gamma-Alpha
# For each cutoff in [0.2, 0.4, 0.8], plot edges with Jaccard ≥ cutoff
# Higher Jaccard -> more saturated red (A01821-based gradient)
# FIX: color scale is ALWAYS 0.2 -> 1.0 (does NOT change with cutoff)

import math
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.path import Path
from matplotlib.patches import PathPatch, Circle
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
import seaborn as sns
# ──────────────────────────────────────────────────────
# 1. Read & clean matrices
# ──────────────────────────────────────────────────────
file_paths = {
    "Amino Acids": "../../../data/graphing/NIS/average_similarity_Amino_AcidsM9.xlsx",
    "Cofactors":   "../../../data/graphing/NIS/average_similarity_CofactorsM9.xlsx",
    "Nucleotides": "../../../data/graphing/NIS/average_similarity_NucleotidesM9.xlsx",
    "Lipids":      "../../../data/graphing/NIS/average_similarity_LipidsM9.xlsx",
}

def clean_labels(df: pd.DataFrame) -> pd.DataFrame:
    """Strip 'Thermo…' and '…Vitconstrained' parts from labels."""
    return df.rename(
        index=lambda s: (s.replace("Thermo", "")
                           .replace("SuccinateVitconstrained", "")
                           .replace("XyloseVitconstrained", "")),
        columns=lambda s: (s.replace("Thermo", "")
                             .replace("SuccinateVitconstrained", "")
                             .replace("XyloseVitconstrained", "")),
    )

missing_codes = {"mip", "cohm"}

def drop_missing(df: pd.DataFrame) -> pd.DataFrame:
    keep = [
        lab for lab in df.index
        if lab[:4] not in missing_codes and lab[:3] not in missing_codes
    ]
    return df.loc[keep, keep]

matrices = {
    name: drop_missing(clean_labels(pd.read_excel(path, index_col=0)))
    for name, path in file_paths.items()
}

# ──────────────────────────────────────────────────────
# 2. Phylum mapping & order
# ──────────────────────────────────────────────────────
phyla_dict = {
    "cohm": "Firmicutes", "cuvk": "Actinobacteria", "cxy":  "Actinobacteria",
    "mip":  "Actinobacteria", "rfa":  "Actinobacteria", "cpi": "Bacteroidetes",
    "flk":  "Bacteroidetes",  "mgk":  "Bacteroidetes",  "bbet": "Alphaproteobacteria",
    "buol": "Betaproteobacteria", "cauj": "Alphaproteobacteria", "dri": "Alphaproteobacteria",
    "lgu":  "Gammaproteobacteria", "lur":  "Gammaproteobacteria", "mlot": "Alphaproteobacteria",
    "pmor": "Gammaproteobacteria", "pscf": "Gammaproteobacteria", "pzu":  "Alphaproteobacteria",
    "racd": "Gammaproteobacteria", "tro":  "Alphaproteobacteria", "vpe":  "Betaproteobacteria",
}

phyla_order = [
    "Firmicutes",
    "Betaproteobacteria",
    "Bacteroidetes",
    "Actinobacteria",
    "Gammaproteobacteria",
    "Alphaproteobacteria",
    "Unknown",
]

palette      = sns.color_palette("Set2", len(phyla_order))
phyla_colors = {phy: palette[i] for i, phy in enumerate(phyla_order)}

def phylum_of(label: str) -> str:
    return phyla_dict.get(label[:4], "Unknown")

# ──────────────────────────────────────────────────────
# 3. Common labels & circular ordering
# ──────────────────────────────────────────────────────
label_sets = [set(df.index) for df in matrices.values()]
common_labels = sorted(set.intersection(*label_sets))
if not common_labels:
    raise ValueError("No common labels found across all matrices after cleaning.")

ordered_labels = sorted(
    common_labels,
    key=lambda lab: (phyla_order.index(phylum_of(lab)), lab),
)

n = len(ordered_labels)
angles = {lab: 2 * math.pi * i / n for i, lab in enumerate(ordered_labels)}

# ──────────────────────────────────────────────────────
# 4. Helper functions & base colormap
# ──────────────────────────────────────────────────────
cutoffs = [0.2, 0.4, 0.6,0.8]     # thresholds we want
edge_lw = 1.5                 # constant stroke width

def polar_to_cartesian(r, theta):
    return r * math.cos(theta), r * math.sin(theta)

# Build a sequential red scale from very light → A01821
base_red = "#453B36"
cmap_edges = sns.light_palette(base_red, n_colors=256, as_cmap=True)

# ──────────────────────────────────────────────────────
# 4b. FIXED normalization for ALL plots
# ──────────────────────────────────────────────────────
GLOBAL_VMIN = 0.2
GLOBAL_VMAX = 1.0
norm = mcolors.Normalize(vmin=GLOBAL_VMIN, vmax=GLOBAL_VMAX)

# ──────────────────────────────────────────────────────
# 5. Loop over cutoffs and metrics
# ──────────────────────────────────────────────────────
R_nodes = 1.0
R_control = 0.7

for cutoff in cutoffs:
    for metric_name, df in matrices.items():
        df = df.loc[ordered_labels, ordered_labels]

        edges = []
        for i, src in enumerate(ordered_labels):
            for j in range(i + 1, n):
                tgt = ordered_labels[j]
                val = df.iloc[i, j]
                if pd.notna(val) and val >= cutoff:
                    edges.append((src, tgt, float(val)))

        print(f"cutoff={cutoff}, {metric_name}: {len(edges)} edges with Jaccard ≥ {cutoff}")

        fig, ax = plt.subplots(figsize=(10, 10))
        ax.set_aspect("equal")
        ax.axis("off")

        # outer circle
        circle = Circle((0, 0), R_nodes,
                        edgecolor="black", facecolor="none",
                        linewidth=1, zorder=1)
        ax.add_patch(circle)

        # nodes
        for lab in ordered_labels:
            theta = angles[lab]
            x, y = polar_to_cartesian(R_nodes, theta)
            ax.scatter(
                x, y,
                s=20,
                color=phyla_colors[phylum_of(lab)],
                edgecolor="black",
                linewidth=0.5,
                zorder=5,
            )

        # labels
        for lab in ordered_labels:
            theta = angles[lab]
            x, y = polar_to_cartesian(R_nodes * 1.08, theta)
            rot = math.degrees(theta)
            if 90 < rot < 270:
                rot += 180
                ha = "right"
            else:
                ha = "left"

            ax.text(
                x, y, lab,
                fontsize=6,
                rotation=rot,
                rotation_mode="anchor",
                ha=ha,
                va="center",
                zorder=6,
            )

        # edges coloured by Jaccard (fixed 0.2 → 1.0 mapping)
        for src, tgt, val in edges:
            theta_src = angles[src]
            theta_tgt = angles[tgt]
            x0, y0 = polar_to_cartesian(R_nodes, theta_src)
            x1, y1 = polar_to_cartesian(R_nodes, theta_tgt)

            theta_mid = (theta_src + theta_tgt) / 2.0
            if abs(theta_src - theta_tgt) > math.pi:
                theta_mid += math.pi
            x_ctrl, y_ctrl = polar_to_cartesian(R_control, theta_mid)

            verts = [(x0, y0), (x_ctrl, y_ctrl), (x1, y1)]
            codes = [Path.MOVETO, Path.CURVE3, Path.CURVE3]

            color = cmap_edges(norm(val))

            path = Path(verts, codes)
            patch = PathPatch(
                path,
                lw=edge_lw,
                edgecolor=color,
                facecolor="none",
                alpha=1.0,
                zorder=2,
            )
            ax.add_patch(patch)

        # Phylum legend
        phylum_handles = [
            Line2D(
                [0], [0],
                marker="o", linestyle="",
                color=phyla_colors[phy],
                label=phy,
                markersize=6,
            )
            for phy in phyla_order
        ]
        phylum_leg = ax.legend(
            handles=phylum_handles,
            title="Phylum",
            loc="upper right",
            bbox_to_anchor=(0.98, 0.98),
            frameon=False,
        )
        ax.add_artist(phylum_leg)

        # Colourbar for Jaccard (ALWAYS 0.2 → 1.0)
        sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap_edges)
        